# 04 Results Analysis

Use this notebook to inspect saved experiment CSV files, compare cleaning conditions, and build report-ready tables.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from IPython.display import Image, display

from scripts.make_analysis_tables import make_all_analysis_tables
from scripts.make_summary_tables import summarize_results
from scripts.make_plots import add_plot_labels, plot_metric
from src.utils.config import load_experiment_config
from src.utils.io import load_dataframe, save_dataframe
from src.utils.training import verify_training_record
import pandas as pd

RESULTS_PATH = Path('results/tables/main_experiment_results.csv')
CONFIG_PATH = Path('configs/experiment_config.yaml')
if not RESULTS_PATH.exists():
    RESULTS_PATH = Path('results/tables/notebook_experiment_results.csv')
    CONFIG_PATH = Path('results/logs/notebook_config.json')

verify_training_record(load_experiment_config(CONFIG_PATH), RESULTS_PATH, PROJECT_ROOT)
df = load_dataframe(RESULTS_PATH)
summary = summarize_results(df)
analysis = make_all_analysis_tables(df)
print(f'Results path: {RESULTS_PATH}')
print(f'Result rows: {len(df):,}')
display(df.head())

## Matrix Coverage

In [ ]:
combo_cols = ['dataset', 'noise_type', 'noise_rate', 'model', 'random_state']
coverage = {
    'datasets': sorted(df['dataset'].unique()),
    'noise_types': sorted(df['noise_type'].unique()),
    'noise_rates': sorted(df['noise_rate'].unique()),
    'models': sorted(df['model'].unique()),
    'seeds': sorted(df['random_state'].unique()),
    'completed_combinations': len(df[combo_cols].drop_duplicates()),
}

display(pd.DataFrame([coverage]).T.rename(columns={0: 'value'}))
display(df.groupby(['dataset', 'condition']).size().unstack(fill_value=0))

## Overall Condition Means

In [ ]:
condition_means = analysis['condition_summary']
display(condition_means)

## Detector Ranking

In [ ]:
detector_ranking = analysis['detector_ranking']
display(detector_ranking)

## Cleaning Gains Over Noisy Baseline

In [ ]:
gain_summary = analysis['cleaning_gain_summary']
display(gain_summary)
display(analysis['paired_cleaning_comparisons'])
display(analysis['combined_detector_comparisons'])

## Accuracy Curves

In [ ]:
PLOT_DATASET = 'dry_bean' if 'dry_bean' in set(df['dataset']) else df['dataset'].iloc[0]
setting = df[df['dataset'].eq(PLOT_DATASET)].iloc[0]
PLOT_MODEL = setting['model']
PLOT_NOISE_TYPE = setting['noise_type']
figure_dir = Path('results/figures/notebook_analysis')
plot_metric(add_plot_labels(df), 'accuracy', PLOT_DATASET, PLOT_NOISE_TYPE, PLOT_MODEL, figure_dir)
display(Image(filename=str(figure_dir / f'{PLOT_DATASET}_{PLOT_NOISE_TYPE}_{PLOT_MODEL}_accuracy.png')))

## Save Report-Friendly Tables

In [ ]:
analysis_dir = Path('results/tables/notebook_analysis')
for name, table in analysis.items():
    save_dataframe(table, analysis_dir / f'{name}.csv')
print(f'Saved analysis tables to {analysis_dir}')